In [ ]:
import pandas as pd
import pypsa

# Part A: Analysis of the loaded network

## Import network

In [ ]:
# Base case
n0 = pypsa.Network("results/tyndp/NT/networks/pre-1107/solved_2050.nc")

# +100 Mt sequestration limit
# n0 = pypsa.Network("results/tyndp/NT/networks/solved_2050.nc")

## Buses

In [ ]:
n0.buses.carrier.unique()

In [ ]:
n0.buses[n0.buses.carrier == "co2"]

In [ ]:
n0.buses[n0.buses.carrier == "co2 stored"]

In [ ]:
n0.buses[n0.buses.carrier == "co2 sequestered"]

## Statistics

### Biomass must run condition & balance

In [ ]:
n0.generators[n0.generators.carrier == "solid biomass"].e_sum_min.div(1e6).round(
    1
)  # in TWh

In [ ]:
n0.generators[n0.generators.carrier == "solid biomass"].e_sum_max.div(1e6).round(
    1
)  # in TWh

In [ ]:
n0.statistics.energy_balance(bus_carrier="solid biomass").div(1e6).round(1).sort_values(
    ascending=False
)

### CO2 balance

In [ ]:
n0.statistics.energy_balance(bus_carrier="co2").div(1e6).round(1).sort_values(
    ascending=False
)

In [ ]:
n0.stores[n0.stores.carrier == "co2"]

Note: the "co2" store is the atmosphere, where "co2" is released into

### CO2 store balance

In [ ]:
n0.statistics.energy_balance(bus_carrier="co2 stored").div(1e6).round(2).sort_values(
    ascending=False
)

Insight: The sequestration is "filled up" with BtL CC and BtG CC (due to high co2 price earning)

In [ ]:
n0.statistics.energy_balance(bus_carrier="co2 sequestered").div(1e6).round(
    2
).sort_values(ascending=False)

In [ ]:
n0.links[n0.links.carrier == "biomass to liquid"]

### SMR CC links

In [ ]:
n0.links[n0.links.carrier == "SMR CC"].p_nom

In [ ]:
n0.statistics.energy_balance(bus_carrier="H2", components="Link").div(1e6).round(2)

### Hydrogen and gas price

In [ ]:
n0.statistics.prices(bus_carrier="H2", weighting="load")
# n0.statistics.prices(bus_carrier="H2", groupby="bus_carrier")
# n0.statistics.prices(bus_carrier="H2", groupby_time=False)

In [ ]:
n0.statistics.energy_balance(bus_carrier="H2").div(1e6).round(2).sort_values(
    ascending=False
)

Gas price insights

In [ ]:
n0.statistics.prices(bus_carrier="gas", weighting="load")

In [ ]:
n0.statistics.energy_balance(bus_carrier="gas").div(1e6).round(2).sort_values(
    ascending=False
)

In [ ]:
n0.generators[n0.generators.carrier == "gas"].marginal_cost

### CO2 and CO2 store price

In [ ]:
n0.statistics.prices(bus_carrier="co2", weighting="load")

In [ ]:
n0.statistics.prices(bus_carrier="co2 stored", weighting="load")

### Synfuels balance

In [ ]:
n0.statistics.energy_balance(bus_carrier="oil").div(1e6).round(2).sort_values(
    ascending=False
)

In [ ]:
n1.links.loc[n1.links.carrier == "biomass to liquid CC"]

In [ ]:
n1.links.carrier.unique()

### Total system cost

In [ ]:
n0.objective / 1e9

# Part B: Custom constraints, adjustments and reoptimisation

Work on a copy so `n0` stays untouched for the comparison in Part C.

In [ ]:
n1 = n0.copy()

### Adjust SMR marginal costs

In [ ]:
# n1.links.loc[n1.links.carrier == "SMR CC", "marginal_cost"] = -60

In [ ]:
# n1.links.loc[n1.links.carrier == "SMR CC", "marginal_cost"]

### Adjust Electrolyzer marginal costs

In [ ]:
# n1.links.loc[n1.links.carrier == "H2 Electrolysis", "marginal_cost"] = (
#     100  # compansate low H2 cost on 168H runs
# )

### Adjust co2 storage quantity

In [ ]:
# Adjust the co2 sequestration limit
n1.global_constraints.loc["co2_sequestration_limit", "constant"] = -199e6

### Adjust synfuel limit 

In [ ]:
synfuel_factor_test = 1

### Adjust fossil gas price

In [ ]:
# Adjust fossil gas price
# n1.generators.loc[n1.generators.carrier == "gas", "marginal_cost"] = 35

## Add constraint on BECCS and reoptimise (synfuel + BECCS constraint)

In [ ]:
def extra(n: pypsa.Network, sns: pd.Index) -> None:
    w = n.snapshot_weightings.generators.loc[sns]
    p = n.model["Link-p"].loc[sns]
    links = n.links
    bio = [("biogas to gas CC", "efficiency2"), ("biomass to liquid CC", "efficiency3")]
    beccs = [
        (p.loc[:, i] * links.loc[i, eff] * w).sum()
        for c, eff in bio
        for i in [links.index[links.carrier == c]]
    ]
    # BECCS limit: 85.78 Mt CO2
    # n.model.add_constraints(beccs[0] + beccs[1] <= 85.78e6, name="beccs_limit")
    syn = links.index[links.carrier.isin(["H2 to e-liquids", "H2 to sng"])]
    ccu = (p.loc[:, syn] * -links.loc[syn, "efficiency2"] * w).sum()
    n.model.add_constraints(
        ccu
        <= n.global_constraints.at["co2_synfuels_limit", "constant"]
        * synfuel_factor_test,
        name="co2_synfuels",
    )


n1.optimize(solver_name="highs", extra_functionality=extra)

# Part C: Comparison of the initial and the reoptimised network

Each table shows `initial` (`n0`), `reoptimised` (`n1`) and their difference.

In [ ]:
def compare(f) -> pd.DataFrame:
    df = pd.concat(
        {"initial": pd.Series(f(n0)), "reoptimised": pd.Series(f(n1))}, axis=1
    ).fillna(0)
    return df.assign(diff=df.reoptimised - df.initial)

## Biomass balance

In [ ]:
compare(
    lambda n: (
        n.statistics.energy_balance(bus_carrier="solid biomass")
        .div(1e6)
        .round(1)
        .sort_values(ascending=False)
    )
)

## CO2 balance

In [ ]:
compare(
    lambda n: (
        n.statistics.energy_balance(bus_carrier="co2")
        .div(1e6)
        .round(1)
        .sort_values(ascending=False)
    )
)

## CO2 store balance

In [ ]:
compare(
    lambda n: (
        n.statistics.energy_balance(bus_carrier="co2 stored")
        .div(1e6)
        .round(2)
        .sort_values(ascending=False)
    )
)

In [ ]:
compare(
    lambda n: (
        n.statistics.energy_balance(bus_carrier="co2 sequestered")
        .div(1e6)
        .round(2)
        .sort_values(ascending=False)
    )
)

In [ ]:
compare(lambda n: n.links.loc[n.links.carrier == "biomass to liquid", "p_nom_opt"])

In [ ]:
compare(lambda n: n.buses_t.marginal_price.loc[:, "co2 stored"].mean())

## SMR CC links

In [ ]:
compare(lambda n: n.links.loc[n.links.carrier == "SMR CC", "p_nom_opt"])

In [ ]:
compare(
    lambda n: (
        n.statistics.energy_balance(bus_carrier="H2", components="Link")
        .div(1e6)
        .round(2)
    )
)

## Hydrogen and gas price

In [ ]:
compare(lambda n: n.statistics.prices(bus_carrier="H2", weighting="load"))

In [ ]:
compare(
    lambda n: (
        n.statistics.energy_balance(bus_carrier="H2")
        .div(1e6)
        .round(2)
        .sort_values(ascending=False)
    )
)

In [ ]:
compare(lambda n: n.statistics.prices(bus_carrier="gas", weighting="load"))

In [ ]:
compare(
    lambda n: (
        n.statistics.energy_balance(bus_carrier="gas")
        .div(1e6)
        .round(2)
        .sort_values(ascending=False)
    )
)

## CO2 and CO2 store price

In [ ]:
compare(lambda n: n.statistics.prices(bus_carrier="co2", weighting="load"))

In [ ]:
compare(lambda n: n.statistics.prices(bus_carrier="co2 stored", weighting="load"))

## Synfuels balance

In [ ]:
compare(
    lambda n: (
        n.statistics.energy_balance(bus_carrier="oil")
        .div(1e6)
        .round(2)
        .sort_values(ascending=False)
    )
)

In [ ]:
compare(
    lambda n: n.statistics.energy_balance(bus_carrier="e-liquids").div(1e6).round(1)
)

## Total system cost

In [ ]:
compare(lambda n: n.objective / 1e9)

# Test area

In [ ]:
n1.links[n1.links.carrier == "H2 Electrolysis"]

In [ ]:
n1.links[n1.links.carrier == "SMR CC"]